# 👾 Unidad 3 — Deep Q-Learning con Atari

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### Una unidad diferente

En los notebooks anteriores implementamos todo desde cero.  
Este notebook es distinto: usamos **RL-Baselines3-Zoo**, una herramienta que ya tiene
DQN implementado y optimizado para Atari.

El objetivo aquí **no es implementar**, sino **entender y analizar**:

| Lo que haremos | Por qué es valioso |
|----------------|-------------------|
| Analizar el YAML de hiperparámetros de DQN | Conectar la teoría del Cap. 3.2 con la práctica |
| Lanzar el entrenamiento con un comando | Saber usar herramientas reales de RL |
| Leer e interpretar los logs | Entender qué ocurre durante el entrenamiento |
| Analizar el comportamiento del agente | Desarrollar intuición sobre qué aprende una CNN |
| Experimentar con otro juego de Atari | Verificar que el mismo enfoque generaliza |

> ⚠️ **GPU obligatoria.** A diferencia de los notebooks anteriores, aquí la GPU no es opcional.
> Sin ella el entrenamiento tarda varias horas. Actívala antes de continuar.

> ⏱️ **Tiempo estimado:** ~2.5 horas (la mayor parte es esperar el entrenamiento)  
> Para un resultado rápido (no certificable): ~15 minutos con `--n-timesteps 50000`

---
## 0 · Primero lo primero: ¿qué vamos a lograr?

Antes de escribir nada, veamos el resultado de un agente DQN entrenado en Space Invaders:

Observa:
- Cómo el agente se mueve lateralmente para esquivar disparos
- Cómo prioriza las filas superiores (valen más puntos)
- Cómo no cruza ciertos límites — aprendió la geometría del juego solo

Nada de esto fue programado explícitamente. Emergió del entrenamiento.

In [ ]:
%%html
<video controls autoplay loop style="width:100%; max-width:600px; border-radius:8px;">
  <source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4"
          type="video/mp4">
</video>
<p style="color:gray; font-size:0.85em;">Agente DQN entrenado en SpaceInvadersNoFrameskip-v4</p>

---
## 1 · Configurar Colab

### 1.1 · Activar la GPU — OBLIGATORIO ⚡

DQN con Atari requiere procesar millones de frames de imagen. Sin GPU:

| Configuración | Tiempo para 1M pasos |
|--------------|---------------------|
| Sin GPU (CPU) | ~6–8 horas |
| GPU T4 | ~1.5–2 horas |
| GPU A100 (Colab Pro) | ~40 min |

**Pasos:**
1. Menú: **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
2. Seleccionar **GPU T4**
3. Guardar

Verifica que la GPU está activa ejecutando la siguiente celda:

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU detectada: {result.stdout.strip()}")
else:
    print("❌ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4")

### 1.2 · Instalar RL-Baselines3-Zoo

**RL-Baselines3-Zoo** es una colección de scripts y configuraciones de entrenamiento
construida sobre Stable-Baselines3. Incluye:
- Hiperparámetros pre-tuneados para cientos de entornos
- Scripts de entrenamiento, evaluación y publicación en el Hub
- Wrappers automáticos para Atari (preprocesamiento de imágenes)

> ⚠️ La instalación puede mostrar un aviso de conflicto de dependencias —
> es normal y no afecta el funcionamiento.

In [ ]:
# Instalar RL-Baselines3-Zoo desde GitHub
!pip install -q git+https://github.com/DLR-RM/rl-baselines3-zoo

# Dependencias del sistema
!apt-get install -q swig cmake ffmpeg

print("✅ RL-Baselines3-Zoo instalado")

### 1.3 · Instalar ROMs de Atari

Los juegos de Atari requieren archivos ROM — son los datos del juego original.  
Gymnasium los descarga automáticamente una vez que aceptas la licencia:

- `gymnasium[atari]` — los entornos de Atari para Gymnasium
- `gymnasium[accept-rom-license]` — acepta la licencia para descargar los ROMs
- `AutoROM` — herramienta que instala los ROMs en el lugar correcto

In [ ]:
!pip install -q 'gymnasium[atari]' 'gymnasium[accept-rom-license]'
!pip install -q autorom
!AutoROM --accept-license -q

print("✅ ROMs de Atari instalados")

### 1.4 · Pantalla virtual

In [ ]:
!apt install -q python-opengl xvfb
!pip install -q pyvirtualdisplay

from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("✅ Pantalla virtual iniciada")

---
## 2 · Antes de entrenar: ¿qué configuración usará DQN?

Antes de lanzar el entrenamiento, vamos a abrir y analizar el archivo YAML que
RL-Zoo usa para DQN con entornos Atari. Este archivo contiene **todos los hiperparámetros**
que estudiamos en las slides del Capítulo 3.2 — ahora los veremos con valores reales.

In [ ]:
# Descargar el archivo de configuración de DQN para Atari
!wget -q https://raw.githubusercontent.com/DLR-RM/rl-baselines3-zoo/master/hyperparams/dqn.yml -O dqn.yml

# Mostrar solo la sección 'atari'
with open('dqn.yml') as f:
    contenido = f.read()

# Extraer la sección atari
inicio = contenido.find('atari:')
fin = contenido.find('\n\n', inicio)
seccion_atari = contenido[inicio:fin]
print("═══ Configuración DQN para Atari (dqn.yml) ═══")
print(seccion_atari)

### Decodificando el YAML — conexión con la teoría

Cada parámetro que ves tiene una razón de ser. Aquí lo conectamos con lo que estudiamos:

| Parámetro YAML | Valor | Conexión con la teoría (Cap. 3.x) |
|----------------|-------|----------------------------------|
| `AtariWrapper` | activado | Preprocesamiento automático: escala de grises, recorte, 84×84 |
| `frame_stack: 4` | 4 frames | Cap. 3.3 — apilamos 4 frames para capturar movimiento |
| `policy: CnnPolicy` | CNN | Cap. 3.2 — red convolucional que procesa imágenes |
| `n_timesteps: 1e7` | 10M pasos | DQN necesita mucha experiencia para aprender Atari |
| `buffer_size: 100000` | 100K | Cap. 3.2 — tamaño del búfer de repetición (Experience Replay) |
| `learning_starts: 100000` | 100K pasos | Primero llenamos el búfer antes de empezar a entrenar |
| `target_update_interval: 1000` | cada 1000 pasos | Cap. 3.2 — cada cuántos pasos se congela la red objetivo |
| `batch_size: 32` | 32 | Tamaño del mini-lote para el gradiente |
| `exploration_fraction: 0.1` | 10% del entrenamiento | ε decae durante el 10% inicial de los pasos |
| `exploration_final_eps: 0.01` | ε mínimo = 1% | Siempre habrá un 1% de exploración |

### ¿Por qué `SpaceInvadersNoFrameskip-v4` y no `SpaceInvaders`?

Hay dos versiones del entorno:

| Versión | Comportamiento |
|---------|----------------|
| `SpaceInvaders-v4` | Salta frames automáticamente — la red ve menos información |
| `SpaceInvadersNoFrameskip-v4` | **Sin saltar frames** — la red ve cada frame del juego |

Con `NoFrameskip` el agente recibe más información y el `AtariWrapper` aplica el frame stacking
de forma controlada — como vimos en el Cap. 3.3.

---
## 3 · Entrenar el agente DQN en Space Invaders

### El comando de entrenamiento

RL-Zoo entrena con un solo comando de terminal. Aquí lo desglosamos:

```bash
python -m rl_zoo3.train
    --algo dqn                          # algoritmo: Deep Q-Network
    --env SpaceInvadersNoFrameskip-v4   # entorno de Atari
    -f logs/                            # carpeta donde guardar el modelo
    -c dqn.yml                          # archivo de hiperparámetros
```

### ¿Cuánto tiempo tardará?

El YAML usa `n_timesteps: 1e7` (10 millones de pasos) para resultados de calidad.
En Colab con GPU T4, 10M pasos tarda ~2 horas. Hay dos opciones:

| Opción | Pasos | Tiempo | Resultado |
|--------|-------|--------|-----------|
| Prueba rápida | 50,000 | ~5 min | El agente aprende algo pero no es bueno |
| Certificación HF | 1,000,000+ | ~1–2 h | Puntuación competitiva |

> 📌 **Recomendación:** empieza con 50,000 pasos para verificar que todo funciona,
> luego lanza el entrenamiento completo.

### Guardar una copia en Google Drive (importante)

Colab puede desconectarse durante un entrenamiento largo. Monta tu Drive para no perder el progreso:

In [ ]:
# Montar Google Drive para no perder el modelo si Colab se desconecta
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/rl-course/unit3', exist_ok=True)
print("✅ Google Drive montado en /content/drive/MyDrive/rl-course/unit3")

### Paso 1: Entrenamiento de prueba rápida (~5 min)

Ejecuta esto primero para verificar que la instalación funciona correctamente.
Con 50,000 pasos el agente no aprende bien, pero podemos ver que el pipeline funciona.

In [ ]:
# Entrenamiento rápido — solo para verificar que funciona
# El agente no aprenderá bien con tan pocos pasos
!python -m rl_zoo3.train \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    -f logs/ \
    -c dqn.yml \
    --n-timesteps 50000

print("\n✅ Entrenamiento de prueba completado")
print("Ahora puedes lanzar el entrenamiento completo en la siguiente celda.")

### Paso 2: Entrenamiento completo (~1-2 horas con GPU T4)

> ☕ Una vez que lances esta celda, el entrenamiento corre solo.
> Puedes cerrar la pestaña y volver más tarde — el modelo se guarda en `logs/`.
> Si Colab se desconecta, el progreso **se pierde** a menos que hayas montado Drive.

In [ ]:
# Entrenamiento completo para la certificación
# Quita el comentario cuando estés listo para el entrenamiento largo

# !python -m rl_zoo3.train \
#     --algo dqn \
#     --env SpaceInvadersNoFrameskip-v4 \
#     -f logs/ \
#     -c dqn.yml

print("Descomenta las líneas de arriba para lanzar el entrenamiento completo.")
print("Recuerda: tarda ~1-2 horas con GPU T4.")

---
## 4 · Leer los logs — ¿qué está pasando?

Durante el entrenamiento, RL-Zoo imprime estadísticas periódicamente.  
Aprender a leerlas es una habilidad importante para cualquier proyecto de RL.

### Ejemplo de log real

```
-------------------------------
| rollout/              |          |
|    ep_len_mean        | 812      |  ← duración media de un episodio (frames)
|    ep_rew_mean        | 187      |  ← PUNTUACIÓN MEDIA — lo más importante
|    exploration_rate   | 0.753    |  ← ε actual (sigue bajando)
| time/                 |          |
|    fps                | 234      |  ← frames procesados por segundo
|    iterations         | 1        |
|    time_elapsed       | 429      |  ← segundos desde el inicio
|    total_timesteps    | 1024     |  ← pasos totales completados
| train/                |          |
|    learning_rate      | 0.0001   |  ← α actual
|    loss               | 0.0342   |  ← pérdida de la red (debería bajar)
|    n_updates          | 256      |  ← actualizaciones del gradiente hechas
-------------------------------
```

### Lo que hay que observar

| Métrica | Al inicio | Cuando aprende | ¿Preocupante si...? |
|---------|-----------|----------------|---------------------|
| `ep_rew_mean` | Negativo o muy bajo | Sube gradualmente | No sube después de 500k pasos |
| `exploration_rate` | ~1.0 | Baja hasta 0.01 | Se queda en 1.0 (el decay no funciona) |
| `loss` | Alta, variable | Baja y se estabiliza | Sube continuamente (divergencia) |
| `fps` | — | 200-500 con GPU | < 50 (falta GPU) |

> 📌 **Consejo:** no te preocupes si `ep_rew_mean` sube y baja irregularmente al principio.
> Es completamente normal — el agente alterna entre explorar y explotar.
> La tendencia general a largo plazo es lo que importa.

In [ ]:
# Ver los archivos generados por el entrenamiento
import os

if os.path.exists('logs/dqn/SpaceInvadersNoFrameskip-v4_1'):
    archivos = os.listdir('logs/dqn/SpaceInvadersNoFrameskip-v4_1')
    print("Archivos generados por el entrenamiento:")
    for f in sorted(archivos):
        print(f"  {f}")
else:
    print("El entrenamiento aún no ha creado archivos en logs/")
    print("Ejecuta la celda de entrenamiento primero.")

---
## 5 · Evaluar el agente entrenado

### El comando de evaluación

Una vez entrenado, evaluamos el agente con `rl_zoo3.enjoy`:

```bash
python -m rl_zoo3.enjoy
    --algo dqn
    --env SpaceInvadersNoFrameskip-v4
    --no-render                  # sin ventana gráfica (Colab no tiene pantalla)
    --n-timesteps 5000           # pasos de evaluación
    --folder logs/               # carpeta donde está el modelo guardado
```

### ¿Qué puntuación es buena?

Para tener contexto, aquí hay puntuaciones de referencia en Space Invaders:

| Agente | Puntuación media |
|--------|-----------------|
| Humano experto | ~1,668 |
| DQN paper original (DeepMind, 2015) | ~1,976 |
| DQN con 10M pasos en Colab | ~200–600 |
| DQN con 50k pasos (prueba rápida) | ~100–200 |
| Agente aleatorio | ~148 |

> 🏆 **Meta de certificación HF:** puntuación media ≥ 200 puntos.

In [ ]:
# Evaluar el agente entrenado en 5000 pasos
!python -m rl_zoo3.enjoy \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    --no-render \
    --n-timesteps 5000 \
    --folder logs/

### Grabar un video del agente jugando

Podemos grabar un video replay para visualizarlo en el notebook:

In [ ]:
# Grabar video del agente (quita el comentario después de entrenar)
# !python -m rl_zoo3.enjoy \
#     --algo dqn \
#     --env SpaceInvadersNoFrameskip-v4 \
#     --n-timesteps 1000 \
#     --folder logs/ \
#     --record-video \
#     --video-folder videos/

# Ver el video si fue grabado
import os
from IPython.display import Video

video_dir = 'videos'
if os.path.exists(video_dir):
    videos = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    if videos:
        video_path = os.path.join(video_dir, sorted(videos)[-1])
        print(f"Mostrando video: {video_path}")
        display(Video(video_path, embed=True, width=500))
    else:
        print("No se encontraron videos. Descomenta las líneas de arriba para grabarlos.")
else:
    print("Directorio de videos no encontrado.")

---
## 6 · Analizar un agente ya entrenado — BeamRider 🚀

Esta es la sección más interesante del notebook.

Vamos a descargar un agente DQN **pre-entrenado por el equipo de SB3** en BeamRider
y analizaremos su comportamiento detalladamente.

BeamRider es un juego diferente de Space Invaders:
- La nave se mueve lateralmente disparando a enemigos
- Hay un límite de disparos — no puedes spamear
- Los enemigos tienen patrones de movimiento distintos

Primero, el video del agente:

In [ ]:
%%html
<video controls autoplay loop style="width:100%; max-width:600px; border-radius:8px;">
  <source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4"
          type="video/mp4">
</video>
<p style="color:gray; font-size:0.85em;">Agente DQN pre-entrenado en BeamRiderNoFrameskip-v4 · SB3 Team</p>

### Descargar y evaluar el modelo pre-entrenado

In [ ]:
# Descargar el modelo pre-entrenado de SB3 desde el Hub
!python -m rl_zoo3.load_from_hub \
    --algo dqn \
    --env BeamRiderNoFrameskip-v4 \
    -orga sb3 \
    -f rl_trained/

print("✅ Modelo pre-entrenado descargado en rl_trained/")

In [ ]:
# Evaluar el modelo pre-entrenado
!python -m rl_zoo3.enjoy \
    --algo dqn \
    --env BeamRiderNoFrameskip-v4 \
    --no-render \
    --n-timesteps 5000 \
    --folder rl_trained/

### Análisis de comportamiento — preguntas de reflexión

Después de ver el video del agente, reflexiona sobre estas preguntas.  
No hay respuestas únicas — el objetivo es desarrollar intuición sobre qué aprende DQN.

**1. ¿Qué información visual usa el agente?**

Recuerda que el agente ve 4 frames de 84×84 píxeles en escala de grises.  
¿Qué patrones detecta la CNN que le permiten comportarse como lo hace?

**2. ¿Qué estrategias emergieron sin ser programadas?**

Observa el video y anota al menos 2 comportamientos específicos que el agente
aprendió. Por ejemplo: ¿evita ciertos bordes? ¿dispara en ciertos momentos?

**3. El límite de disparos en BeamRider**

BeamRider penaliza disparar demasiado rápido. ¿Cómo afecta esto a la estrategia
del agente en comparación con Space Invaders donde puedes disparar libremente?

**4. Limitaciones visibles**

¿En qué situaciones parece confundirse el agente? ¿Qué tipos de errores comete?  
¿Podrías relacionarlos con las limitaciones de la Q-table o de la CNN?

**5. Frame stacking en acción**

Sin frame stacking, el agente vería solo una imagen estática y no podría detectar
el movimiento de los proyectiles. ¿Puedes ver evidencia en el video de que el agente
*sí* detecta la trayectoria de los disparos enemigos?

In [ ]:
# Espacio para tus notas de análisis
# (celda de texto — escribe aquí tus observaciones)

mis_observaciones = '''
1. Estrategia observada 1: ...
2. Estrategia observada 2: ...
3. Limitaciones notadas: ...
4. Pregunta que me surge: ...
'''

print("Mis observaciones del agente BeamRider:")
print(mis_observaciones)

---
## 7 · 🏋️ Ejercicio: entrena tu agente en otro juego de Atari

### El único ejercicio de este notebook

Ya viste el proceso completo con Space Invaders. Ahora te toca a ti aplicarlo
a un juego diferente.

**Elige uno de estos entornos:**

| Juego | Nombre del entorno | Descripción | Dificultad |
|-------|-------------------|-------------|------------|
| Pong | `PongNoFrameskip-v4` | Paleta vs. paleta — el más fácil | ⭐ |
| Breakout | `BreakoutNoFrameskip-v4` | Romper ladrillos con una pelota | ⭐⭐ |
| Pacman | `MsPacmanNoFrameskip-v4` | Laberinto clásico con fantasmas | ⭐⭐⭐ |
| Asteroids | `AsteroidsNoFrameskip-v4` | Nave espacial destruyendo asteroides | ⭐⭐⭐ |
| Qbert | `QbertNoFrameskip-v4` | Saltar sobre cubos cambiando colores | ⭐⭐⭐⭐ |

**Pasos:**
1. Escoge un juego de la tabla
2. Cambia el nombre del entorno en los comandos de abajo
3. Entrena (~50k pasos para prueba, 1M+ para buenos resultados)
4. Evalúa y compara con Space Invaders
5. Reflexiona: ¿qué fue diferente? ¿Qué aprendió el agente?

In [ ]:
# 🏋️ EJERCICIO — cambia el entorno y ejecuta

# Elige tu entorno aquí:
MI_ENTORNO = 'PongNoFrameskip-v4'  # ← cambia esto

print(f"Entorno elegido: {MI_ENTORNO}")
print(f"Lanza el entrenamiento con:")
print(f"  python -m rl_zoo3.train --algo dqn --env {MI_ENTORNO} -f logs_exp/ -c dqn.yml --n-timesteps 50000")

In [ ]:
# Entrenamiento — descomenta y ejecuta
# !python -m rl_zoo3.train \
#     --algo dqn \
#     --env {MI_ENTORNO} \
#     -f logs_exp/ \
#     -c dqn.yml \
#     --n-timesteps 50000   # aumenta para mejores resultados

import os
# Para ejecutar el entrenamiento con la variable definida arriba:
os.system(f'python -m rl_zoo3.train --algo dqn --env {MI_ENTORNO} -f logs_exp/ -c dqn.yml --n-timesteps 50000')

In [ ]:
# Evaluar tu agente en el segundo juego
import os
!python -m rl_zoo3.enjoy \
    --algo dqn \
    --env {MI_ENTORNO} \
    --no-render \
    --n-timesteps 2000 \
    --folder logs_exp/

In [ ]:
# Espacio para comparar resultados
resultados = {
    'Space Invaders (entrenamiento de prueba)': '???',  # rellena con tu resultado
    MI_ENTORNO: '???',                                  # rellena con tu resultado
}

print("Comparativa de resultados:")
for juego, puntuacion in resultados.items():
    print(f"  {juego}: {puntuacion}")

print()
print("Preguntas de reflexión:")
print("  ¿Qué juego fue más fácil de aprender? ¿Por qué?")
print("  ¿Qué estrategia aprendió el agente en tu juego elegido?")
print("  ¿Necesitó más o menos pasos que Space Invaders para obtener resultados similares?")

---

# 🔒 Sección opcional — Publicar en el Hub

> Requiere cuenta de HuggingFace. Completamente opcional.

---

## [OPCIONAL] Publicar tu agente en el Hugging Face Hub

Una vez que tengas un modelo entrenado con buena puntuación, puedes publicarlo.
RL-Zoo lo hace en un solo comando que evalúa, graba video y sube todo.

### Paso 1: Iniciar sesión

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Paso 2: Publicar Space Invaders

Reemplaza `TU_USUARIO` con tu nombre de usuario de HuggingFace:

In [ ]:
# Publicar el modelo de Space Invaders
# Reemplaza TU_USUARIO con tu nombre de usuario de HuggingFace
!python -m rl_zoo3.push_to_hub \
    --algo dqn \
    --env SpaceInvadersNoFrameskip-v4 \
    --repo-name dqn-SpaceInvadersNoFrameskip-v4 \
    --orga TU_USUARIO \
    -f logs/

### Paso 3: Publicar tu segundo juego (si lo entrenaste)

In [ ]:
# Publicar el modelo de tu segundo juego
# !python -m rl_zoo3.push_to_hub \
#     --algo dqn \
#     --env {MI_ENTORNO} \
#     --repo-name dqn-mi-juego \
#     --orga TU_USUARIO \
#     -f logs_exp/

---

## 🎉 ¡Notebook completado!

En este notebook hiciste algo diferente a los anteriores: en vez de implementar,  
**aprendiste a usar herramientas reales de RL** que se usan en investigación y producción.

Lo que cubriste:

1. **Analizaste** el archivo YAML de hiperparámetros — conectando teoría y práctica
2. **Entrenaste** un agente DQN con RL-Baselines3-Zoo en Space Invaders
3. **Leíste** los logs de entrenamiento y sabes qué significa cada métrica
4. **Evaluaste** el desempeño con métricas cuantitativas
5. **Analizaste** el comportamiento de un agente pre-entrenado en BeamRider
6. **Experimentaste** con un segundo juego de Atari a tu elección

### ¿Qué sigue?

- 📊 **Módulo 4:** Dejamos los métodos de valor y aprendemos a optimizar
  directamente la política — el algoritmo REINFORCE
- 📓 **Notebook 4:** REINFORCE desde cero en PyTorch con CartPole y PixelCopter

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*